<a href="https://colab.research.google.com/github/Janya22/mcp-chain/blob/main/Feasibility1_clean.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U bitsandbytes
!pip install -U transformers sentence-transformers
!pip install -U accelerate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 15.2 MB/s eta 0:00:00


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

model_name = "mistralai/Mistral-7B-Instruct-v0.2"
quantization_config = BitsAndBytesConfig(load_in_8bit=True)

# Use `dtype` not `torch_dtype`
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=quantization_config,  # Modern quantization setup
    dtype=torch.float16
)



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

model-00001-of-00003.safetensors:   0%|          | 0.00/4.94G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Minimal example tool registry
tool_registry = [
    {
        "tool_name": "SearchHotel",
        "description": "Finds available hotels based on location, price, and rating.",
        "input_parameters": ["location", "min_price", "max_price"]
    },
    {
        "tool_name": "BookHotel",
        "description": "Books hotel room given hotel name, dates, and guest info.",
        "input_parameters": ["hotel_name", "check_in", "check_out", "adults", "children"]
    },
    {
        "tool_name": "SearchFlight",
        "description": "Finds available flights given source, target, dates.",
        "input_parameters": ["origin", "destination", "date"]
    },
    {
        "tool_name": "BookFlight",
        "description": "Books a flight for a specified route and date.",
        "input_parameters": ["origin", "destination", "date", "passenger_name"]
    }
]

# Precompute dense embeddings for descriptions
embedder = SentenceTransformer('all-MiniLM-L6-v2')
descriptions = [tool['description'] for tool in tool_registry]
tool_embeddings = embedder.encode(descriptions)

def retrieve_tools(query, k=2):
    """Return top-k similar tools to query"""
    qemb = embedder.encode([query])
    sims = np.dot(tool_embeddings, qemb.T).flatten()
    return [tool_registry[i] for i in np.argsort(sims)[-k:][::-1]]

def invoke_tool(tool_name, step_description):
    """
    Simulate execution of an external tool.
    In practice, this might be an API call. For now, just return a placeholder.
    """
    return f"Simulated {tool_name} response for step: '{step_description}'"



In [ ]:
import torch

def llm_generate(prompt, model, tokenizer, max_new_tokens=128):
    device = model.device
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


def autonomous_tool_chain(
    user_query, model, tokenizer, k_tools=2, max_retries=2, verbose=True
):
    print(f"User Query: {user_query}\n")
    # 1. Plan decomposition (with clarification prompt)
    plan_prompt = (
        f'Given the user query: "{user_query}"\n'
        "Break down the problem into a sequence of numbered, atomic steps that may require external tools.\n"
        "For ambiguous, multi-part, or multi-tool queries, include each required tool step explicitly.\nPlan:"
    )
    plan = llm_generate(plan_prompt, model, tokenizer, max_new_tokens=128)
    if verbose:
        print("[PLAN GENERATED]")
        print(plan)

    lines = [x.strip() for x in plan.split('\n') if x.strip() and x[0].isdigit()]
    step_to_tool = []
    thoughts = []

    for step in lines:
        step_txt = step[2:].strip() if step[1] == '.' else step
        for attempt in range(max_retries):
            candidates = retrieve_tools(step_txt, k=k_tools)
            tool_list = ', '.join([f"{t['tool_name']}" for t in candidates])
            tool_prompt = (
                f"Step: {step_txt}\n"
                f"Available tools: {tool_list}\n"
                "Which tool is most appropriate, and why? If none are adequate, say 'NONE' and explain."
            )
            response = llm_generate(tool_prompt, model, tokenizer, max_new_tokens=48)
            tool_choice = None
            for t in candidates:
                if t['tool_name'] in response:
                    tool_choice = t['tool_name']
                    break

            # Optional: invoke tool and get output
            tool_output = None
            if tool_choice:
                tool_output = invoke_tool(tool_choice, step_txt)

            critique_prompt = (
                f"Step: '{step_txt}'\n"
                f"Tool chosen: {tool_choice}\n"
                f"Tool output: {tool_output}\n"
                "Is this step complete and correct? If not, what should be retried or replanned?"
            )
            critique = llm_generate(critique_prompt, model, tokenizer, max_new_tokens=32)
            if "complete" in critique.lower() or attempt == max_retries - 1:
                break
            else:
                # Optionally rephrase or replan
                step_txt = critique + " (retrying step)"
            thoughts.append((response, critique))

        step_to_tool.append((step_txt, tool_choice, response))
        if verbose:
            print(f"\n[TOOL CHOICE FOR STEP]: {step_txt}")
            print(f"> Candidates: {tool_list}")
            print(f"> LLM: {response}")
            print(f"> Critique: {critique}")

    if verbose:
        print("\n[AGENT CHAIN OF THOUGHT]")
        for step, tool, reasoning in step_to_tool:
            print(f"- Step: '{step}' → Tool: '{tool}' (Reason: {reasoning})")

    tool_invocations = [(tool, step) for step, tool, _ in step_to_tool if tool]
    print(f"\n[FINAL TOOL INVOCATION SEQUENCE]")
    for tool, step in tool_invocations:
        print(f"Call tool: {tool} for: '{step}'")

    return plan, step_to_tool, tool_invocations, thoughts




In [ ]:
# Example chain query (tests multi-step, cross-tool workflow)
test_query = "Find affordable hotels in Miami for June 10-15, book one room for two adults, and then book a flight from NYC to Miami for those dates."
plan, step_to_tool, tool_invocations, thoughts = autonomous_tool_chain(
    test_query,
    model,
    tokenizer,
    k_tools=2,
    verbose=True
)


User Query: Find affordable hotels in Miami for June 10-15, book one room for two adults, and then book a flight from NYC to Miami for those dates.

[PLAN GENERATED]
Given the user query: "Find affordable hotels in Miami for June 10-15, book one room for two adults, and then book a flight from NYC to Miami for those dates."
Break down the problem into a sequence of numbered, atomic steps that may require external tools.
For ambiguous, multi-part, or multi-tool queries, include each required tool step explicitly.
Plan:
1. Set up a travel search engine account or log in, if necessary.
2. Use the search engine to find affordable hotels in Miami for two adults from June 10-15.
   a. Set the number of guests to 2 and the dates to June 10-15.
   b. Filter the results by affordability or price.
3. Select a hotel and book a room for two adults from June 10-15.
4. Use a flight search engine to find and book a round-trip flight for two adults from NYC to Miami for

[TOOL CHOICE FOR STEP]: Set up

In [8]:
from nbformat import read, write, NO_CONVERT

with open("Feasibility1.ipynb", "r", encoding="utf-8") as f:
    nb = read(f, NO_CONVERT)

# Remove widget metadata safely
nb.metadata.pop("widgets", None)

with open("Feasibility1_clean.ipynb", "w", encoding="utf-8") as f:
    write(nb, f)


FileNotFoundError: [Errno 2] No such file or directory: 'Feasibility1.ipynb'

In [12]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [13]:
!ls "/content/drive/MyDrive/Colab Notebooks"


API-bank_data_conv.ipynb  NEW-Trial.ipynb		     Untitled16.ipynb
api_bank_server.ipynb	  SearchL1.ipynb		     Untitled17.ipynb
blip_finetuned.ipynb	  Search_pdt.ipynb		     Untitled1.ipynb
blip.ipynb		  SearchTerm_mapping_combined.ipynb  Untitled2.ipynb
fashion.ipynb		  SearchTerm_mapping_pdtname.ipynb   Untitled3.ipynb
Feasibility1.ipynb	  Untitled0.ipynb		     Untitled4.ipynb
Feasibility.ipynb	  Untitled10.ipynb		     Untitled5.ipynb
flan_50.ipynb		  Untitled11.ipynb		     Untitled6.ipynb
flan_T5_tester.ipynb	  Untitled12.ipynb		     Untitled7.ipynb
image_tester.ipynb	  Untitled13.ipynb		     Untitled8.ipynb
Lab1AA.ipynb		  Untitled14.ipynb		     Untitled9.ipynb
Lab2AA.ipynb		  Untitled15.ipynb


In [14]:
from nbformat import read, write, NO_CONVERT

input_path = "/content/drive/MyDrive/Colab Notebooks/Feasibility1.ipynb"
output_path = "/content/drive/MyDrive/Colab Notebooks/Feasibility1_clean.ipynb"

with open(input_path, "r", encoding="utf-8") as f:
    nb = read(f, NO_CONVERT)

nb.metadata.pop("widgets", None)

with open(output_path, "w", encoding="utf-8") as f:
    write(nb, f)


In [15]:
!ls "/content/drive/MyDrive/Colab Notebooks"

API-bank_data_conv.ipynb  Lab2AA.ipynb			     Untitled15.ipynb
api_bank_server.ipynb	  NEW-Trial.ipynb		     Untitled16.ipynb
blip_finetuned.ipynb	  SearchL1.ipynb		     Untitled17.ipynb
blip.ipynb		  Search_pdt.ipynb		     Untitled1.ipynb
fashion.ipynb		  SearchTerm_mapping_combined.ipynb  Untitled2.ipynb
Feasibility1_clean.ipynb  SearchTerm_mapping_pdtname.ipynb   Untitled3.ipynb
Feasibility1.ipynb	  Untitled0.ipynb		     Untitled4.ipynb
Feasibility.ipynb	  Untitled10.ipynb		     Untitled5.ipynb
flan_50.ipynb		  Untitled11.ipynb		     Untitled6.ipynb
flan_T5_tester.ipynb	  Untitled12.ipynb		     Untitled7.ipynb
image_tester.ipynb	  Untitled13.ipynb		     Untitled8.ipynb
Lab1AA.ipynb		  Untitled14.ipynb		     Untitled9.ipynb
